# Official GPT Training on Google Colab

> **Note:** If Colab tells you "Restart Runtime Required" during dependency installation, simply click "Restart" and then run the remaining cells again. Your cloned files and dataset will still be there.


## 1. Check GPU & CUDA

In [ ]:
# Verify GPU is attached
!nvidia-smi

# Verify PyTorch sees CUDA correctly
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repo

In [ ]:
import os
if not os.path.exists('LLM'):
    !git clone https://github.com/Harshkumar2306/LLM.git
%cd LLM

## 4. Checkout Branch

In [ ]:
# Set the branch you want to experiment with
# Examples:
# BRANCH = "exp/rope"
# BRANCH = "exp/kv-caching"
BRANCH = "main"

!git fetch origin
!git checkout {BRANCH}

## 5. Install Requirements

In [ ]:
!pip install -q -r requirements.txt

## 6. Prepare Dataset

In [ ]:
!mkdir -p data
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O data/input.txt
!python scripts/prepare_data.py --input data/input.txt --outdir data/

## 7. Print Configuration & Save Metadata

In [ ]:
import subprocess
commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print("Training on branch:", BRANCH)
print("Commit Hash:", commit_hash)

import json
!mkdir -p results
with open("results/metadata.json", "w") as f:
    json.dump({"branch": BRANCH, "commit": commit_hash}, f)

# Set the experiment output directory based on the branch
EXPERIMENT_DIR = f"/content/drive/MyDrive/LLM-Experiments/{BRANCH.replace('exp/', '')}"

## 8. Train

In [ ]:
!python scripts/train.py \
    --train_bin data/train.bin \
    --val_bin data/val.bin \
    --device cuda \
    --batch_size 4 \
    --grad_accum_steps 4 \
    --out_dir {EXPERIMENT_DIR}

## 9. Evaluate

In [ ]:
!python scripts/evaluate.py \
    --checkpoint {EXPERIMENT_DIR}/best.pt \
    --val_bin data/val.bin \
    --batch_size 4 \
    --device cuda

## 10. Benchmark

In [ ]:
!python scripts/benchmark.py \
    --batch_size 4 \
    --device cuda

## 11. Generate Text

In [ ]:
!python scripts/generate.py \
    --checkpoint {EXPERIMENT_DIR}/best.pt \
    --prompt "ROMEO:" \
    --device cuda

## 12. Save Results

In [ ]:
# This will automatically push the benchmark/evaluation results in `results/` to your current branch
!git config --global user.email "colab@example.com"
!git config --global user.name "Google Colab"
!git add results/
!git commit -m "Save Colab benchmark and evaluation results for {BRANCH}"
# Note: Pushing requires authentication. You can either use a Personal Access Token in the clone URL
# or manually push from your local machine after downloading the results directory.
!git push origin {BRANCH}